# AutoML-M06: Comparativa Final

**TFM: Predicción de Abandono Universitario**

| | |
|---|---|
| **Autora** | María José Morte |
| **Email** | mjmorteruiz@uoc.edu (UOC) \| morte@uji.es (UJI) |

---

## 🎯 Objetivo

Comparativa final de **todos los frameworks AutoML** (M01-M05) para
**Caso D** (alerta temprana) y **Caso D_strict** (producción).
HTML interactivo con filtros por caso.

## 📥 Entrada
Todos los `data/automl/results_*.parquet` generados por M01-M05.

## 📦 Genera
- `data/automl/automl_comparativa_final.xlsx` — Excel con hojas por caso
- `data/automl/automl_comparativa_final.json` — JSON para uso programático
- `data/automl/automl_top_modelos.parquet` — Mejor modelo por framework × caso
- `docs/html/fase_automl/m06_comparativa.html` — HTML interactivo

## ⚠️ Kernel: `tfm_abandono` (entorno base del proyecto)

In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN
# ============================================================================
# Qué hace: Carga librerías y configuración del proyecto.
# Requisitos: Kernel tfm_abandono (entorno base).
# ============================================================================

import sys, warnings, subprocess
from pathlib import Path
warnings.filterwarnings('ignore')

# --- Auto-verificación de dependencias ---
DEPENDENCIAS_REQUERIDAS = {
    'seaborn': 'seaborn',
    'matplotlib': 'matplotlib',
    'sklearn': 'scikit-learn',
    'pandas': 'pandas',
    'numpy': 'numpy',
    'pyarrow': 'pyarrow',
    'jinja2': 'jinja2',
    'openpyxl': 'openpyxl',
}

faltantes = []
for modulo, paquete_pip in DEPENDENCIAS_REQUERIDAS.items():
    try:
        __import__(modulo)
    except ImportError:
        faltantes.append(paquete_pip)

if faltantes:
    print(f'⚠️ Instalando dependencias faltantes: {faltantes}')
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '--quiet'] + faltantes
    )
    print(f'✅ Instaladas: {faltantes}')
else:
    print('✅ Todas las dependencias disponibles')

# --- Detección de entorno ---
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/AU_UJI')
else:
    ROOT = Path.cwd()
    for _ in range(5):
        if (ROOT / 'src').exists(): break
        ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from src.config import RUTA_AUTOML, RUTA_HTML, info_entorno
from src.utils import crear_directorios, formato_numero_es
from src.utils.graficos import figura_a_base64
from src.html import generar_kpis_html, generar_seccion_html, generar_html_navegacion_completa, guardar_html
from src.html.render import render_base_html

RUTA_FASE_AUTOML_HTML = RUTA_HTML / 'fase_automl'
crear_directorios([RUTA_FASE_AUTOML_HTML])
fmt = formato_numero_es

info_entorno()

✅ Todas las dependencias disponibles
✓ Directorios verificados: 1
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw

In [2]:
# ============================================================================
# CELDA 2: CARGAR TODOS LOS RESULTADOS
# ============================================================================
# Qué hace: Carga y concatena todos los results_*.parquet de M01-M05.
# Genera: df_all con todas las métricas de todos los frameworks.
# ============================================================================

print('='*70)
print('AUTOML-M06: COMPARATIVA FINAL')
print('='*70)

results_files = sorted(RUTA_AUTOML.glob('results_*.parquet'))
print(f'\nArchivos encontrados: {len(results_files)}')

dfs = []
for f in results_files:
    df = pd.read_parquet(f)
    dfs.append(df)
    print(f'  ✅ {f.name}: {len(df)} filas')

if not dfs:
    raise FileNotFoundError('No hay results_*.parquet. Ejecutar M01-M05 primero.')

df_all = pd.concat(dfs, ignore_index=True)

# Verificar esquema
cols_requeridas = ['caso', 'framework', 'model_name', 'f1', 'auc_roc', 'mcc']
cols_faltantes = [c for c in cols_requeridas if c not in df_all.columns]
if cols_faltantes:
    print(f'⚠️ Columnas faltantes: {cols_faltantes}')
else:
    print(f'✅ Esquema unificado verificado')

print(f'\n📊 Total: {len(df_all)} resultados')
for caso in sorted(df_all['caso'].unique()):
    n = (df_all['caso']==caso).sum()
    print(f'  Caso {caso}: {n} modelos')
print(f'Frameworks: {list(df_all["framework"].unique())}')

AUTOML-M06: COMPARATIVA FINAL

Archivos encontrados: 5
  ✅ results_autogluon.parquet: 28 filas
  ✅ results_baselines.parquet: 12 filas
  ✅ results_h2o.parquet: 44 filas
  ✅ results_lazypredict.parquet: 54 filas
  ✅ results_pycaret.parquet: 30 filas
✅ Esquema unificado verificado

📊 Total: 168 resultados
  Caso D: 84 modelos
  Caso D_strict: 84 modelos
Frameworks: ['AutoGluon', 'baselines', 'H2O', 'lazypredict', 'PyCaret']


In [3]:
# ============================================================================
# CELDA 3: RANKINGS Y GRÁFICOS
# ============================================================================
# Qué hace: Genera rankings por caso y gráficos comparativos.
# ============================================================================

graficos_b64 = {}

# --- Colores por framework ---
COLORES_FW = {
    'baselines': '#3182ce',
    'lazypredict': '#d69e2e',
    'PyCaret': '#38a169',
    'H2O': '#805ad5',
    'AutoGluon': '#ed8936',
}

# --- Ranking por caso ---
for caso in sorted(df_all['caso'].unique()):
    df_c = df_all[df_all['caso']==caso].sort_values('f1', ascending=False)
    # Excluir Dummy para el ranking
    df_c_real = df_c[~df_c['model_name'].str.startswith('Dummy')]
    print(f'\n--- TOP 10 CASO {caso} (por F1) ---')
    print(df_c_real[['framework', 'model_name', 'f1', 'auc_roc', 'mcc', 'train_time_sec']].head(10).to_string(index=False))

    # Gráfico: Top 10 barras horizontales con color por framework
    df_top = df_c_real.head(10).copy()
    df_top = df_top.sort_values('f1', ascending=True)
    fig, ax = plt.subplots(figsize=(12, 7))
    y_pos = np.arange(len(df_top))
    colores = [COLORES_FW.get(fw, '#718096') for fw in df_top['framework']]
    bars = ax.barh(y_pos, df_top['f1'], color=colores, alpha=0.85, height=0.6)
    ax.scatter(df_top['auc_roc'], y_pos, color='#e53e3e', s=50, zorder=5, label='AUC-ROC')
    ax.set_yticks(y_pos)
    ax.set_yticklabels([f"{r['framework']}: {r['model_name']}" for _, r in df_top.iterrows()], fontsize=9)
    ax.set_xlabel('Score')
    ax.set_title(f'Comparativa Caso {caso}: Top 10 Modelos (todos los frameworks)', fontweight='bold', fontsize=14)
    ax.legend(fontsize=9)
    ax.set_xlim(0, 1.05)
    for bar, val in zip(bars, df_top['f1']):
        ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=8, color='#2d3748')
    plt.tight_layout()
    graficos_b64[f'top_{caso}'] = figura_a_base64(fig)
    plt.close()

# --- Gráfico D vs D_strict: Mejor F1 por framework ---
casos = sorted(df_all['caso'].unique())
frameworks = sorted(df_all['framework'].unique())
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(frameworks))
w = 0.35
colores_caso = {'D': '#3182ce', 'D_strict': '#805ad5'}

for i, caso in enumerate(casos):
    vals = []
    for fw in frameworks:
        mask = (df_all['caso']==caso) & (df_all['framework']==fw) & (~df_all['model_name'].str.startswith('Dummy'))
        if mask.sum() > 0:
            vals.append(df_all[mask]['f1'].max())
        else:
            vals.append(0)
    ax.bar(x + i*w, vals, w, label=f'Caso {caso}', color=colores_caso.get(caso, '#718096'))

ax.set_xticks(x + w/2)
ax.set_xticklabels(frameworks, rotation=20, ha='right', fontsize=10)
ax.set_ylabel('Mejor F1')
ax.set_title('Caso D vs D_strict: Mejor F1 por Framework', fontweight='bold', fontsize=14)
ax.legend(fontsize=10)
ax.set_ylim(0, 1.05)
ax.axhline(y=0.8, color='gray', ls='--', alpha=0.3)
plt.tight_layout()
graficos_b64['d_vs_dstrict'] = figura_a_base64(fig)
plt.close()

print(f'\n✅ {len(graficos_b64)} gráficos generados')


--- TOP 10 CASO D (por F1) ---
framework                  model_name       f1  auc_roc      mcc  train_time_sec
AutoGluon             CatBoost_BAG_L2 0.813764 0.951177 0.743291           44.16
AutoGluon           LightGBMXT_BAG_L2 0.813054 0.950925 0.742305           44.16
AutoGluon         WeightedEnsemble_L3 0.810994 0.951071 0.740591           44.16
AutoGluon             LightGBM_BAG_L2 0.810936 0.950534 0.739520           44.16
AutoGluon             CatBoost_BAG_L1 0.810140 0.949350 0.740844           44.16
      H2O                       GBM_4 0.809524 0.945329 0.733146           13.07
AutoGluon         WeightedEnsemble_L2 0.808572 0.950399 0.739003           44.16
AutoGluon     RandomForestGini_BAG_L2 0.808564 0.948843 0.737153           44.16
      H2O StackedEnsemble_AllModels_1 0.808283 0.947615 0.733933           13.07
      H2O                    GBM_grid 0.807652 0.944629 0.731600           13.07

--- TOP 10 CASO D_strict (por F1) ---
framework                     model_na

In [4]:
# ============================================================================
# CELDA 4: GUARDAR EXCEL, JSON Y PARQUET RESUMEN
# ============================================================================
# Qué hace: Exporta resultados en múltiples formatos.
# Genera: Excel (hojas por caso), JSON, Parquet top modelos.
# ============================================================================

# --- Excel con hojas por caso ---
ruta_xlsx = RUTA_AUTOML / 'automl_comparativa_final.xlsx'
with pd.ExcelWriter(ruta_xlsx, engine='openpyxl') as writer:
    df_all.to_excel(writer, sheet_name='todos', index=False)
    for caso in sorted(df_all['caso'].unique()):
        df_all[df_all['caso']==caso].sort_values('f1', ascending=False).to_excel(
            writer, sheet_name=f'caso_{caso}', index=False)
    # Resumen: mejor por framework × caso
    resumen = df_all.loc[df_all.groupby(['caso', 'framework'])['f1'].idxmax()]
    resumen.sort_values(['caso', 'f1'], ascending=[True, False]).to_excel(
        writer, sheet_name='resumen', index=False)
print(f'📊 Excel: {ruta_xlsx.name}')

# --- JSON ---
ruta_json = RUTA_AUTOML / 'automl_comparativa_final.json'
df_all.to_json(ruta_json, orient='records', force_ascii=False)
print(f'📊 JSON: {ruta_json.name}')

# --- Parquet resumen (mejor por framework × caso) ---
ruta_top = RUTA_AUTOML / 'automl_top_modelos.parquet'
resumen.to_parquet(ruta_top, index=False)
print(f'📊 Top modelos: {ruta_top.name}')

📊 Excel: automl_comparativa_final.xlsx
📊 JSON: automl_comparativa_final.json
📊 Top modelos: automl_top_modelos.parquet


In [5]:
# ============================================================================
# CELDA 5: GENERAR HTML CON FILTROS INTERACTIVOS
# ============================================================================
# Qué hace: Genera HTML con botones de filtro D/D_strict/Todo,
#   tabla completa, gráficos y conclusión.
# Genera: docs/html/fase_automl/m06_comparativa.html
# ============================================================================

nav_fases, nav_modulos = generar_html_navegacion_completa(
    fase_activa='fase_automl', modulo_activo='m06'
)

# --- KPIs ---
mejor_global = df_all[~df_all['model_name'].str.startswith('Dummy')].sort_values('f1', ascending=False).iloc[0]
n_frameworks = len(df_all['framework'].unique())
KPIS = [
    {'valor': str(len(df_all)), 'titulo': 'Total Modelos'},
    {'valor': str(n_frameworks), 'titulo': 'Frameworks'},
    {'valor': f'{mejor_global["f1"]:.4f}', 'titulo': 'Mejor F1'},
    {'valor': str(mejor_global['model_name'])[:25], 'titulo': 'Mejor Modelo'},
]

# --- Botones de filtro (JavaScript) ---
filtro_js = '''
<div style="margin:20px 0;text-align:center;">
  <button onclick="filtrar('todo')" id="btn_todo" style="padding:10px 25px;margin:5px;border:2px solid #3182ce;background:#3182ce;color:white;border-radius:8px;cursor:pointer;font-weight:bold;font-size:14px;">📋 Todo</button>
  <button onclick="filtrar('D')" id="btn_D" style="padding:10px 25px;margin:5px;border:2px solid #38a169;background:white;color:#38a169;border-radius:8px;cursor:pointer;font-weight:bold;font-size:14px;">🅳 Caso D</button>
  <button onclick="filtrar('D_strict')" id="btn_D_strict" style="padding:10px 25px;margin:5px;border:2px solid #805ad5;background:white;color:#805ad5;border-radius:8px;cursor:pointer;font-weight:bold;font-size:14px;">🅳ₛ Caso D_strict</button>
</div>
<script>
function filtrar(caso) {
  document.querySelectorAll(".fila-resultado").forEach(row => {
    if (caso === "todo" || row.dataset.caso === caso) {
      row.style.display = "";
    } else {
      row.style.display = "none";
    }
  });
  ["todo","D","D_strict"].forEach(c => {
    var btn = document.getElementById("btn_" + c);
    if (btn) {
      if (c === caso) {
        btn.style.background = btn.style.borderColor;
        btn.style.color = "white";
      } else {
        btn.style.background = "white";
        btn.style.color = btn.style.borderColor;
      }
    }
  });
  document.querySelectorAll(".graf-caso").forEach(el => {
    if (caso === "todo" || el.dataset.caso === caso || el.dataset.caso === "ambos") {
      el.style.display = "";
    } else {
      el.style.display = "none";
    }
  });
}
</script>
'''

# --- Tabla con data-caso ---
df_sorted = df_all[~df_all['model_name'].str.startswith('Dummy')].sort_values(
    ['caso', 'f1'], ascending=[True, False]
)
tabla = '<table style="width:100%;border-collapse:collapse;">\n'
tabla += '<tr style="background:#2d3748;color:white;">'
for col in ['Caso', 'Framework', 'Modelo', 'Acc', 'F1', 'AUC', 'MCC', 'Prec', 'Recall', 'Tiempo']:
    tabla += f'<th style="padding:8px;text-align:center;font-size:11px;">{col}</th>'
tabla += '</tr>\n'

for i, (idx, row) in enumerate(df_sorted.iterrows()):
    caso = row['caso']
    bg_caso = '#f0fff4' if caso == 'D' else '#f5f0ff'
    bg = bg_caso if i % 2 == 0 else 'white'
    tabla += f'<tr class="fila-resultado" data-caso="{caso}" style="background:{bg};">'
    tabla += f'<td style="padding:6px;text-align:center;font-weight:bold;font-size:11px;">{caso}</td>'
    tabla += f'<td style="padding:6px;text-align:center;font-size:11px;">{row["framework"]}</td>'
    tabla += f'<td style="padding:6px;font-size:11px;">{row["model_name"]}</td>'
    for c in ['accuracy', 'f1', 'auc_roc', 'mcc', 'precision', 'recall']:
        v = row.get(c, 0)
        if pd.isna(v): v = 0
        color = '#38a169' if v > 0.7 else '#ed8936' if v > 0.5 else '#e53e3e'
        tabla += f'<td style="text-align:center;color:{color};font-size:11px;">{v:.4f}</td>'
    t = row.get('train_time_sec', 0)
    tabla += f'<td style="text-align:center;font-size:11px;">{t:.1f}s</td>'
    tabla += '</tr>\n'
tabla += '</table>'

# --- Gráficos con data-caso ---
graf_html = '<div style="display:flex;flex-wrap:wrap;gap:20px;justify-content:center;">'
for key, b64 in graficos_b64.items():
    if key.startswith('top_'):
        caso_g = key.replace('top_', '')
    elif key == 'd_vs_dstrict':
        caso_g = 'ambos'
    else:
        caso_g = 'ambos'
    graf_html += f'<div class="graf-caso" data-caso="{caso_g}" style="flex:1;min-width:400px;">'
    graf_html += f'<img src="data:image/png;base64,{b64}" style="max-width:100%;border-radius:8px;">'
    graf_html += '</div>'
graf_html += '</div>'

# --- Secciones HTML ---
s_filtro = generar_seccion_html('🔍 Filtrar por Caso', filtro_js)
s_tabla = generar_seccion_html('📊 Ranking de Modelos', tabla)
s_graf = generar_seccion_html('📈 Visualizaciones', graf_html)

# --- Conclusión ---
mejor_d = df_all[(df_all['caso']=='D') & (~df_all['model_name'].str.startswith('Dummy'))].sort_values('f1', ascending=False).iloc[0]
mejor_ds = df_all[(df_all['caso']=='D_strict') & (~df_all['model_name'].str.startswith('Dummy'))].sort_values('f1', ascending=False).iloc[0]

s_concl = generar_seccion_html('📌 Conclusión', f'''
<div style="display:grid;grid-template-columns:1fr 1fr;gap:20px;">
  <div style="background:#f0fff4;padding:20px;border-radius:10px;border-left:4px solid #38a169;">
    <h4 style="color:#38a169;margin-top:0;">🅳 Caso D — Alerta Temprana</h4>
    <p><strong>{mejor_d["framework"]}: {mejor_d["model_name"]}</strong></p>
    <p>F1 = {mejor_d["f1"]:.4f} | AUC = {mejor_d["auc_roc"]:.4f} | MCC = {mejor_d["mcc"]:.4f}</p>
  </div>
  <div style="background:#f5f0ff;padding:20px;border-radius:10px;border-left:4px solid #805ad5;">
    <h4 style="color:#805ad5;margin-top:0;">🅳ₛ Caso D_strict — Producción</h4>
    <p><strong>{mejor_ds["framework"]}: {mejor_ds["model_name"]}</strong></p>
    <p>F1 = {mejor_ds["f1"]:.4f} | AUC = {mejor_ds["auc_roc"]:.4f} | MCC = {mejor_ds["mcc"]:.4f}</p>
  </div>
</div>
<div style="background:#EBF8FF;padding:15px;border-radius:10px;margin-top:15px;border-left:4px solid #3182ce;">
  <strong>📝 Nota:</strong> Todos los resultados son sobre datos limpios (sin leakage,
  sin constantes, sin variables traidoras). Los datasets fueron auditados en F3-M08.
</div>
''')

# --- Renderizar HTML ---
html = render_base_html(
    titulo='M06: Comparativa Final', icono='🏆',
    subtitulo='Pre-Modelado AutoML — Caso D + D_strict',
    nav_fases=nav_fases, nav_modulos=nav_modulos,
    contenido=generar_kpis_html(KPIS) + s_filtro + s_tabla + s_graf + s_concl,
    notebook_nombre='fautoml_m06_comparativa.ipynb', notebook_carpeta='fase_automl'
)
ruta_html = RUTA_FASE_AUTOML_HTML / 'm06_comparativa.html'
guardar_html(html, ruta_html)
print(f'\n✅ HTML: {ruta_html}')
print(f'   Incluye: filtros D/D_strict/Todo interactivos')

✅ HTML guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase_automl\m06_comparativa.html

✅ HTML: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase_automl\m06_comparativa.html
   Incluye: filtros D/D_strict/Todo interactivos


In [6]:
# ============================================================================
# CELDA 6: RESUMEN
# ============================================================================

print('\n' + '='*60)
print('✅ AUTOML-M06 COMPLETADO')
print('='*60)
print(f'Total modelos: {len(df_all)}')
print(f'Frameworks: {list(df_all["framework"].unique())}')
print(f'Casos: {list(df_all["caso"].unique())}')
print(f'\n📊 Archivos:')
print(f'   Excel: {ruta_xlsx}')
print(f'   JSON: {ruta_json}')
print(f'   Top modelos: {ruta_top}')
print(f'   HTML: {ruta_html}')
print(f'\n🏆 Mejores modelos:')
print(f'   D: {mejor_d["framework"]} → {mejor_d["model_name"]} (F1={mejor_d["f1"]:.4f})')
print(f'   D_strict: {mejor_ds["framework"]} → {mejor_ds["model_name"]} (F1={mejor_ds["f1"]:.4f})')
print(f'\n✅ Fase AutoML completada')
print('='*60)


✅ AUTOML-M06 COMPLETADO
Total modelos: 168
Frameworks: ['AutoGluon', 'baselines', 'H2O', 'lazypredict', 'PyCaret']
Casos: ['D', 'D_strict']

📊 Archivos:
   Excel: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl\automl_comparativa_final.xlsx
   JSON: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl\automl_comparativa_final.json
   Top modelos: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl\automl_top_modelos.parquet
   HTML: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase_automl\m06_comparativa.html

🏆 Mejores modelos:
   D: AutoGluon → CatBoost_BAG_L2 (F1=0.8138)
   D_strict: AutoGluon → CatBoost_BAG_L2 (F1=0.7970)

✅ Fase AutoML completada
